# ERF chính: 2.000 ảnh và hai kiểm tra độ nhạy

**Câu hỏi:** ResNet tích lũy ảnh hưởng theo độ sâu thế nào? ViT có ảnh hưởng rộng
từ block 3 không? A90 gần nhau ở cuối có đồng nghĩa phân bố giống nhau không?

Đo độ nhạy của **feature trung gian**, không phải class score. Kết quả phụ thuộc
checkpoint, scalar và vị trí. A90 là số pixel tối thiểu chứa 90% khối lượng
gradient, không phải diện tích vùng liên thông hoặc bán kính.

Notebook nhập code dùng chung từ `experiments/erf/`. Bản đã chạy ngày 03/11/2026
và toàn bộ output được giữ ở `experiments/erf/archive/04_erf_2000_20261103.ipynb`.
Notebook này không gắn output lịch sử vào code mới chưa chạy.

## 0. Chuẩn bị

Đồng bộ code project đã cập nhật vào Drive: `experiments/erf/`, `src/`,
`configs/` và manifest. Colab GPU dùng cho gradient; audit/analysis dùng CPU được.
Không cần ảnh/checkpoint để audit CSV. Không tự tải code từ URL.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

IS_COLAB = importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
# Giữ đúng phiên bản thống kê để tái lập việc đọc float và tie ranks của run gốc.
expected_versions = {'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3'}
updates = []
for package, expected in expected_versions.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        updates.append(f'{package}=={expected}')
if updates:
    loaded_before_install = any(name in sys.modules for name in expected_versions)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *updates])
    if loaded_before_install:
        raise RuntimeError('Đã cài bộ thống kê gốc. Restart session/kernel rồi chạy lại từ đầu để nạp đúng thư viện.')
missing = [name for name in ['matplotlib', 'PIL', 'pyarrow', 'tqdm']
           if importlib.util.find_spec(name) is None]
if missing:
    packages = ['pillow' if name == 'PIL' else name for name in missing]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])
import torch
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
PROJECT_ROOT = (Path('/content/drive/MyDrive/cnn-vit-interpretability') if IS_COLAB else
                next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/analysis_set.json').is_file()))
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'data/imagenet/data'
ALLOW_DOWNLOAD = False  # True: cho phép tải đúng checkpoint PyTorch nếu cache trống.
RERUN_MAIN = False     # Giữ False để tận dụng hai run 2.000 ảnh complete.
RUN_BENCHMARK = False  # Đo 50 ảnh trước khi lên lịch compute trên runtime mới.
RUN_SENSITIVITY = False  # Bật khi có GPU/dữ liệu, hoặc điền hai run complete bên dưới.
ABSOLUTE_RUN_DIR = None
QUARTER_RUN_DIR = None

from experiments.erf.erf_experiment import (
    select_main_records, select_sensitivity_records, run_experiment, erf_output_paths,
)
from experiments.erf.erf_workflow import (
    default_run_dirs, new_run_id, audit_main_runs, export_main_analysis,
    run_sensitivity, export_sensitivity,
)
paths = default_run_dirs(PROJECT_ROOT)
OFFICIAL_RUN_DIR = paths['official']
SHARED_RUN_DIR = paths['shared_vit_crop']
PAIRED_RUN_DIR = paths['paired']
records = select_main_records(PROJECT_ROOT, seed=42)
sensitivity_records = select_sensitivity_records(records, seed=42)
print('Primary:', len(records), 'ảnh / 1.000 lớp / 2 ảnh mỗi lớp')
print('Sensitivity:', len(sensitivity_records), 'ID cố định; không lọc đúng/sai')
print('Data shards:', len(list(DATA_DIR.glob('validation-*.parquet'))), '/ 14')

## 1. Định nghĩa phép đo

ResNet-50 V2: layer1/2/3/4; ViT-B/16 V1: block3/6/9/12, trước encoder LN cuối.
Target gốc: vector tại `row=H//2, col=W//2`, scalar $z=\mathrm{mean}(f^2)$.
ERF: $E_{ij}=\sum_c |\partial z/\partial u_{cij}|$, với RGB $u\in[0,1]$
sau resize/crop, trước normalization. Gradient được chia cho std theo chain rule.

A50/A90, dispersion RMS quanh centroid và central/outer mass bổ sung cho nhau.
Stage chỉ là mức sâu tương đối; không chứng minh các tầng tương đương.
Chọn cùng ô trên lưới không đảm bảo cùng receptive center trong ảnh đầu vào.

Official: ResNet resize 232; ViT resize 256; cùng crop 224. Shared crop: cả hai
dùng transform ViT resize 256. Phân tích hai điều kiện riêng.

Cỡ mẫu 2.000 dựa trên pilot SD hiệu số A90 11,127155 điểm %, với biên lập kế hoạch
95% ±0,5 điểm %: N ≈ (1,96 × 11,127155 / 0,5)² ≈ 1.903; chọn 2.000 để có 2 ảnh/lớp.
Đây không phải power analysis hay bảo đảm CI thực tế, không đủ suy luận từng lớp.

## 2. Tận dụng run gốc; chỉ chạy mới khi chủ động bật RERUN_MAIN

Run gốc được kiểm tra từ CSV/JSON. Không suy ra Wilcoxon/CI từ summary đã làm tròn.
Run mới không ghi đè run cũ. Chưa hỗ trợ resume run bị ngắt; checkpoint giữ phần đã tính.

In [ ]:
if RERUN_MAIN:
    if not torch.cuda.is_available():
        raise RuntimeError('Chọn Colab GPU trước khi chạy lại toàn bộ 2.000 ảnh.')
    for mode in ['official', 'shared_vit_crop']:
        directory = erf_output_paths(PROJECT_ROOT, new_run_id(f'main_2000_{mode}'))[0]
        run_experiment(records, DATA_DIR, directory, torch.device('cuda'),
                       preprocessing=mode, project_root=PROJECT_ROOT,
                       allow_download=ALLOW_DOWNLOAD, checkpoint_every=50)
        if mode == 'official':
            OFFICIAL_RUN_DIR = directory
        else:
            SHARED_RUN_DIR = directory
    PAIRED_RUN_DIR = None  # Tính bảng phân tích mới, không so hash với run cũ.
else:
    print('Dùng run complete:')
    print(OFFICIAL_RUN_DIR)
    print(SHARED_RUN_DIR)

In [ ]:
analysis_dir, analysis_figures, result = export_main_analysis(
    PROJECT_ROOT, OFFICIAL_RUN_DIR, SHARED_RUN_DIR, PAIRED_RUN_DIR,
)
print('Audit:', result['checks'])
print('Metrics:', analysis_dir, '| Figures:', analysis_figures)
display(result['tests'])
display(result['crop'])
from IPython.display import Image as DisplayImage
display(DisplayImage(filename=str(analysis_figures / 'paired_mean_ci95.png')))
display(DisplayImage(filename=str(analysis_figures / 'paired_distributions.png')))

2.000 cặp ảnh mỗi phép so sánh; không coi 16.000 dòng là quan sát độc lập.
Bootstrap mean CI95%: 20.000 lần, resample theo cặp ảnh, seed 42; không giữ quota lớp.
Wilcoxon hai phía, round hiệu số 12 chữ số, bỏ zero; Holm trên 16 phép so sánh chính.
CI là pointwise, không hiệu chỉnh đa kiểm định. Wilcoxon không kiểm định trực tiếp
trung bình: CI mean chứa 0 và signed-rank có p nhỏ có thể cùng xảy ra.
Không viết p=0 khi giá trị bị underflow. CI chứa 0 không chứng minh tương đương.

## 3. Benchmark runtime mới (tùy chọn)

50 ảnh lấy từ subset độ nhạy cố định. Metadata ghi thời gian cả I/O và compute,
cùng peak VRAM cho mỗi model. Không dùng số liệu benchmark làm kết quả nghiên cứu.

In [ ]:
if RUN_BENCHMARK:
    benchmark_dir = erf_output_paths(PROJECT_ROOT, new_run_id('benchmark_50'))[0]
    run_experiment(sensitivity_records[:50], DATA_DIR, benchmark_dir,
                   torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                   preprocessing='shared_vit_crop', project_root=PROJECT_ROOT,
                   num_previews=0, allow_download=ALLOW_DOWNLOAD)
    print('Timing / VRAM:', benchmark_dir / 'metadata.json')

## 4. Hai kiểm tra nhỏ trên cùng 2000 ảnh

Chỉ shared crop; giữ weights, layers và seed. Baseline lấy từ metric gốc trên cùng ID.
- `absolute_center`: đổi scalar thành mean(abs(f)), giữ vị trí tâm.
- `energy_quarter`: giữ mean(f²), đổi vị trí thành round(0.25 × (H−1)) mỗi chiều.

Quarter là tọa độ lưới, không phải phép căn chỉnh receptive center chính xác.
Phân tích thay đổi hiệu số ViT−ResNet theo từng ảnh, kèm CI mô tả; không thêm vào
họ 16 kiểm định chính. Không tăng N sau khi thấy p-value. Mặc định chưa chạy mới.

In [ ]:
probe_dirs = {name: Path(path) for name, path in
              [('absolute_center', ABSOLUTE_RUN_DIR), ('energy_quarter', QUARTER_RUN_DIR)] if path is not None}
if RUN_SENSITIVITY:
    if len(probe_dirs) < 2 and not torch.cuda.is_available():
        raise RuntimeError('Cần GPU để chạy hai probe mới; hoặc điền hai run complete để phân tích.')
    probe_dirs = run_sensitivity(PROJECT_ROOT, DATA_DIR, SHARED_RUN_DIR,
                                 existing=probe_dirs, allow_download=ALLOW_DOWNLOAD)
    print('Probe directories:', probe_dirs)
if len(probe_dirs) == 2:
    sensitivity_dir, sensitivity_figures, sensitivity_table = export_sensitivity(
        PROJECT_ROOT, SHARED_RUN_DIR, probe_dirs)
    display(sensitivity_table)
    display(DisplayImage(filename=str(sensitivity_figures / 'sensitivity_gap_change.png')))
    print('Sensitivity:', sensitivity_dir)
else:
    print('PENDING: hai probe chưa chạy đủ; không suy đoán kết quả độ nhạy.')

## 5. Viết và ghép kết quả nhóm

`cross_method_handoff.csv` chọn ví dụ gần median, hai cực trị A90 tầng cuối và
ba preview ngẫu nhiên gốc. Ví dụ chọn theo metric là case study, không phải mẫu đại diện.
Dùng cùng ID/shared crop; Grad-CAM dùng ground-truth class cho cả hai model.
Attention cần ghi rõ query token/head/aggregation; không coi là cùng target ERF.
Thành viên Attention/Grad-CAM bổ sung quan sát/figure và crop hash sau khi tạo crop.
Cột prediction official chỉ là baseline tham khảo, không phải prediction shared crop.

Đọc `report/erf.md` cho kết quả đã xác minh và `docs/erf.md` cho hướng dẫn phối hợp nhóm.
Trạng thái chưa chạy độ nhạy/chưa có kết quả nhóm phải được giữ rõ trong báo cáo.